# T5 TorchAO INT8 CPU Streaming — 5 Texts

Benchmarks only:
- TorchAO INT8 weight-only
- TorchAO dynamic A8W8 INT8

Five fixed inputs of varied length. No quality evaluation.

Measures **actual saved size, TTFT, every inter-token gap, mean/median/p95 ITL, total latency and throughput**. Streaming is implemented with a manual T5 decoder loop using `past_key_values`, so TTFT and ITL timestamps correspond to actual generation steps.

For T5: **TTFT = encoder prefill + first decoder step**. Subsequent tokens reuse KV cache.

In [ ]:
# CELL 0 — install
!pip install -q --no-cache-dir "transformers==5.17.0" "accelerate==1.15.0" "torchao==0.15.0" sentencepiece pandas psutil
print("Installed. Restart once only if Colab requests it.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 135.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 125.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 138.9 MB/s eta 0:00:00
Installed. Restart once only if Colab requests it.


In [ ]:
# CELL 1 — imports/environment
import os,gc,time,shutil
from pathlib import Path
import numpy as np
import pandas as pd
import psutil,torch,transformers,torchao
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM

assert not torch.cuda.is_available(),"Switch Colab to CPU runtime."
CORES=psutil.cpu_count(logical=False) or 2
torch.set_num_threads(CORES)
print("torch",torch.__version__,"transformers",transformers.__version__,"torchao",getattr(torchao,"__version__","?"))
print("cores",CORES,"RAM GB",round(psutil.virtual_memory().total/1024**3,2))

torch 2.11.0+cpu transformers 5.17.0 torchao 0.15.0
cores 1 RAM GB 12.67


In [ ]:
# CELL 2 — config
MODEL_ID="JayShah07/falconai-text-bullet-t5"
MAX_INPUT_LENGTH=2048
MAX_NEW_TOKENS=256
BULLET_TOKEN="<BULLET>"
ROOT=Path("/content/t5_int8_stream")
ARTIFACT_DIR=ROOT/"artifacts"
REPORT_DIR=ROOT/"reports"
ARTIFACT_DIR.mkdir(parents=True,exist_ok=True)
REPORT_DIR.mkdir(parents=True,exist_ok=True)

In [ ]:
# CELL 3 — exact task prompt
TASK_INSTRUCTION="""Convert the following English text into concise bullet points containing all materially important information.

Follow these rules:

- Extract all important and independently useful points.
- The number of bullets must depend entirely on the information in the text.
- Never use a fixed number of bullets.
- Use one bullet for each distinct important point.
- Combine details that naturally belong together.
- Remove repetition, filler, metadata, boilerplate, and trivial details.
- Do not repeat the same information in multiple bullets.
- Preserve important names, dates, numbers, quantities, comparisons, causes, conditions, decisions, and conclusions.
- Do not add, infer, or assume information that is not supported by the source text.
- Do not turn contextual information into new advice or recommendations.
- Keep every bullet concise while preserving the original meaning.
- Return only bullet points.
- Start every bullet with "- ".

Text:"""
def build_text(x): return TASK_INSTRUCTION+"\n"+x.strip()

In [ ]:
# CELL 4 — tokenizer
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,use_fast=True)
assert tokenizer.convert_tokens_to_ids(BULLET_TOKEN)!=tokenizer.unk_token_id
print("vocab",len(tokenizer),"EOS",tokenizer.eos_token_id,"PAD",tokenizer.pad_token_id)

config.json:   0%|          | 0.00/1.55k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

vocab 32101 EOS 1 PAD 0


In [ ]:
# CELL 5 — five varied test texts
TEST_TEXTS={
"01_very_short":"""Apex Systems reported quarterly revenue of $1.8 billion, up 9% year over year. Operating profit increased 6% to $240 million, while management maintained its full-year guidance.""",

"02_short":"""Vertex Systems reported quarterly revenue of $3.8 billion, up 16% from the same period last year. Cloud-service revenue increased 28%, while legacy hardware sales declined 7%. Operating profit rose from $410 million to $475 million, although operating margin decreased from 18.1% to 17.4% because of higher infrastructure and energy costs. The company added 820,000 paying customers and said demand remained strong in North America and Asia.""",

"03_medium":"""Northstar Technologies reported second-quarter revenue of $8.7 billion, an increase of 18% year over year. Growth was driven primarily by cloud infrastructure and enterprise subscriptions. Cloud revenue rose 31%, while the older hardware business declined 9%. Operating profit increased from $940 million to $1.12 billion, but operating margin declined from 21.4% to 20.1% because of higher data-center, energy and hiring costs. The company added 1.6 million subscription customers, taking the total to 12.4 million. Management raised full-year revenue guidance from $33 billion to between $34.5 billion and $35 billion and warned that foreign-exchange movements could reduce fourth-quarter revenue by approximately $300 million.""",

"04_long":"""Meridian Group said annual revenue increased 14% to $21.6 billion as strong growth in digital services offset weaker traditional consulting demand. Digital-services revenue rose 27% and now represents 46% of group sales. Operating profit increased 11% to $2.4 billion, although operating margin slipped from 16.2% to 15.8% after increased spending on data centers, cybersecurity and artificial intelligence. Meridian signed 34 contracts worth more than $100 million, added approximately 18,000 employees primarily in India, Poland and Mexico, and reduced headcount in several higher-cost European offices. Free cash flow rose to $1.9 billion from $1.5 billion. The board approved a 12% dividend increase and a new $2 billion share-repurchase program. Management expects revenue growth of 9% to 11% next year but warned that customers in Germany and France are delaying discretionary technology projects. The company plans two new cloud facilities in Asia and also announced a machine-learning security platform for enterprise customers.""",

"05_very_long":"""Orion Global reported full-year revenue of $47.3 billion, up 19%, after growth across cloud infrastructure, payments and enterprise software. Cloud infrastructure revenue increased 34% to $16.8 billion, supported by financial-services companies and artificial intelligence developers. Payments revenue rose 22% as transaction volume increased 18%, although consumer spending weakened in parts of Western Europe. Enterprise-software revenue increased 11%, while legacy on-premise licensing declined 13%. Operating profit increased from $5.1 billion to $6.0 billion, but operating margin fell from 22.8% to 22.1% because Orion spent heavily on data centers, AI accelerators and security infrastructure. Capital expenditure reached $7.4 billion versus $4.9 billion a year earlier. The company added 4.2 million subscription customers, bringing the total to 31.7 million, while enterprise retention remained above 95%. Orion signed 61 contracts worth more than $50 million, including nine worth more than $250 million each. The company also announced a restructuring of its consumer-device division, eliminating approximately 3,500 positions and consolidating five manufacturing sites into three. The program is expected to cost $420 million to $480 million but generate approximately $650 million in annual savings. Free cash flow increased 16% to $5.8 billion, the quarterly dividend was raised 10%, and another $4 billion was approved for share repurchases. Net debt declined to $8.1 billion from $9.6 billion. Orion expects next-year revenue growth of 12% to 14% and operating margin between 22% and 23%. Risks include weaker European enterprise spending, constrained accelerator supply, higher electricity costs and regulatory reviews of its payments business. Its order backlog is nevertheless 24% higher than a year ago. Finally, Orion introduced an enterprise security product combining automated threat detection, identity monitoring and incident-response tools, with premium customers receiving it from October 15 and general availability scheduled for November 3."""
}
for k,v in TEST_TEXTS.items(): print(k,len(v.split()),"words")

01_very_short 26 words
02_short 66 words
03_medium 102 words
04_long 150 words
05_very_long 288 words


In [ ]:
# ============================================================
# CELL 6 — TORCHAO-SAFE MODEL SIZE HELPERS
# ============================================================

def tensor_nbytes(tensor):

    # TorchAO tensor subclasses may not expose normal storage.
    # numel * element_size works for ordinary tensors and many
    # quantized tensor subclasses without touching data_ptr().
    try:
        return tensor.numel() * tensor.element_size()

    except Exception:
        return 0


def model_tensor_size_mb(model):

    total_bytes = 0

    # Parameters
    for _, param in model.named_parameters():
        total_bytes += tensor_nbytes(param)

    # Buffers
    for _, buffer in model.named_buffers():
        total_bytes += tensor_nbytes(buffer)

    return total_bytes / 1024**2


def model_footprint_mb(model):

    try:
        return model.get_memory_footprint() / 1024**2

    except Exception:
        return model_tensor_size_mb(model)


print("TorchAO-safe size helpers ready.")

TorchAO-safe size helpers ready.


In [ ]:
# ============================================================
# UPDATED STREAMING CELL
# Correct T5/SentencePiece spacing + KV cache + timing
# ============================================================

def stream_generate(
    model,
    text,
    max_new_tokens=MAX_NEW_TOKENS,
    show=True,
):

    # --------------------------------------------------------
    # TOKENIZE SOURCE
    # --------------------------------------------------------

    batch = tokenizer(
        build_text(text),
        return_tensors="pt",
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )

    input_ids = batch["input_ids"]
    attention_mask = batch["attention_mask"]

    input_token_count = int(
        input_ids.shape[-1]
    )


    # --------------------------------------------------------
    # T5 DECODER START TOKEN
    # --------------------------------------------------------

    decoder_start_token_id = (
        model.config.decoder_start_token_id
    )

    if decoder_start_token_id is None:
        decoder_start_token_id = tokenizer.pad_token_id


    decoder_input_ids = torch.tensor(
        [[decoder_start_token_id]],
        dtype=torch.long,
    )


    # --------------------------------------------------------
    # STORAGE
    # --------------------------------------------------------

    generated_ids = []

    # Timestamp when each generated token becomes available.
    token_timestamps = []

    # Actual model-forward duration for every decode step.
    decode_step_times = []

    # Used only for correctly displaying incremental text.
    previous_display_text = ""


    # --------------------------------------------------------
    # START TIMER
    #
    # TTFT includes:
    #   encoder prefill
    #   +
    #   first decoder step
    # --------------------------------------------------------

    total_start = time.perf_counter()


    with torch.inference_mode():

        # ====================================================
        # ENCODER PREFILL — RUN ONCE
        # ====================================================

        encoder_outputs = model.get_encoder()(

            input_ids=input_ids,

            attention_mask=attention_mask,

            return_dict=True,
        )


        past_key_values = None


        # ====================================================
        # DECODER LOOP
        # ====================================================

        for step in range(max_new_tokens):

            step_start = time.perf_counter()


            outputs = model(

                encoder_outputs=encoder_outputs,

                attention_mask=attention_mask,

                decoder_input_ids=decoder_input_ids,

                past_key_values=past_key_values,

                use_cache=True,

                return_dict=True,
            )


            # ------------------------------------------------
            # GREEDY NEXT TOKEN
            # ------------------------------------------------

            logits = outputs.logits[:, -1, :]


            next_token = torch.argmax(

                logits,

                dim=-1,

                keepdim=True,
            )


            token_id = int(
                next_token.item()
            )


            # Timestamp immediately after model produces token.
            token_ready_time = time.perf_counter()


            decode_step_times.append(
                token_ready_time - step_start
            )


            generated_ids.append(
                token_id
            )


            token_timestamps.append(
                token_ready_time
            )


            # =================================================
            # STREAM WITH CORRECT SPACING
            # =================================================
            #
            # DO NOT decode only [token_id].
            #
            # T5 SentencePiece may encode spaces contextually.
            #
            # Instead decode ALL generated tokens so far,
            # then print only the newly-added text.
            # =================================================

            if show:

                visible_ids = [

                    x for x in generated_ids

                    if x not in {

                        tokenizer.pad_token_id,

                        tokenizer.eos_token_id,

                    }
                ]


                raw_cumulative = tokenizer.decode(

                    visible_ids,

                    skip_special_tokens=False,

                    clean_up_tokenization_spaces=False,
                )


                # Convert our special bullet token into
                # visible newline bullet formatting.

                display_text = raw_cumulative.replace(

                    BULLET_TOKEN,

                    "\n- ",
                )


                # ------------------------------------------------
                # PRINT ONLY NEWLY GENERATED TEXT
                # ------------------------------------------------

                if display_text.startswith(
                    previous_display_text
                ):

                    new_text = display_text[
                        len(previous_display_text):
                    ]

                else:

                    # Rare tokenizer normalization fallback:
                    # find common prefix.

                    common = 0

                    max_common = min(
                        len(display_text),
                        len(previous_display_text),
                    )

                    while (
                        common < max_common
                        and
                        display_text[common]
                        ==
                        previous_display_text[common]
                    ):
                        common += 1


                    new_text = display_text[
                        common:
                    ]


                if new_text:

                    print(
                        new_text,
                        end="",
                        flush=True,
                    )


                previous_display_text = (
                    display_text
                )


            # ------------------------------------------------
            # EOS
            # ------------------------------------------------

            if (
                token_id
                ==
                tokenizer.eos_token_id
            ):

                break


            # ------------------------------------------------
            # KV CACHE
            # ------------------------------------------------
            #
            # Next decoder step receives only the latest token.
            # Previous K/V tensors are reused.
            # ------------------------------------------------

            past_key_values = (
                outputs.past_key_values
            )


            decoder_input_ids = (
                next_token
            )


    # --------------------------------------------------------
    # END
    # --------------------------------------------------------

    total_end = time.perf_counter()


    if show:
        print()


    # ========================================================
    # FINAL OUTPUT
    # ========================================================

    visible_ids = [

        x for x in generated_ids

        if x not in {

            tokenizer.pad_token_id,

            tokenizer.eos_token_id,

        }
    ]


    raw_output = tokenizer.decode(

        visible_ids,

        skip_special_tokens=False,

        clean_up_tokenization_spaces=True,
    ).strip()


    if BULLET_TOKEN in raw_output:

        parts = [

            p.strip()

            for p in raw_output.split(
                BULLET_TOKEN
            )

            if p.strip()
        ]


        final_output = "\n".join(

            "- " + p

            for p in parts
        )

    else:

        final_output = raw_output


    # ========================================================
    # TIMING
    # ========================================================

    if token_timestamps:

        ttft_seconds = (

            token_timestamps[0]
            -
            total_start
        )

    else:

        ttft_seconds = np.nan


    # --------------------------------------------------------
    # USER-PERCEIVED TOKEN GAPS
    # --------------------------------------------------------
    #
    # Difference between consecutive completed tokens.
    #
    # Includes very small Python / streaming overhead.
    # This is closest to what a user actually experiences.
    # --------------------------------------------------------

    if len(token_timestamps) >= 2:

        inter_token_seconds = np.diff(
            np.array(
                token_timestamps
            )
        )

    else:

        inter_token_seconds = np.array(
            [],
            dtype=float,
        )


    # --------------------------------------------------------
    # PURE MODEL DECODE STEP TIMES
    # --------------------------------------------------------
    #
    # Excludes notebook print/decode work.
    # First decoder step is excluded because TTFT already
    # measures encoder + first decoder.
    # --------------------------------------------------------

    if len(decode_step_times) >= 2:

        model_decode_itl = np.array(
            decode_step_times[1:]
        )

    else:

        model_decode_itl = np.array(
            [],
            dtype=float,
        )


    overall_latency_seconds = (

        total_end
        -
        total_start
    )


    output_token_count = len(
        generated_ids
    )


    # Decode throughput after first token.

    time_after_first = (

        overall_latency_seconds
        -
        ttft_seconds

        if not np.isnan(ttft_seconds)

        else np.nan
    )


    tokens_after_first = max(

        output_token_count - 1,

        0,
    )


    # ========================================================
    # RETURN
    # ========================================================

    return {

        "output":
            final_output,

        "input_tokens":
            input_token_count,

        "output_tokens":
            output_token_count,


        # ---------------------------
        # FIRST TOKEN
        # ---------------------------

        "ttft_seconds":
            ttft_seconds,


        # ---------------------------
        # USER-PERCEIVED ITL
        # ---------------------------

        "itl_values_ms":
            (
                inter_token_seconds
                *
                1000
            ).tolist(),

        "mean_itl_ms":
            (
                float(
                    np.mean(
                        inter_token_seconds
                    )
                    *
                    1000
                )

                if len(inter_token_seconds)

                else np.nan
            ),

        "median_itl_ms":
            (
                float(
                    np.median(
                        inter_token_seconds
                    )
                    *
                    1000
                )

                if len(inter_token_seconds)

                else np.nan
            ),

        "p95_itl_ms":
            (
                float(
                    np.quantile(
                        inter_token_seconds,
                        0.95,
                    )
                    *
                    1000
                )

                if len(inter_token_seconds)

                else np.nan
            ),


        # ---------------------------
        # PURE MODEL DECODE ITL
        # ---------------------------

        "model_mean_decode_itl_ms":
            (
                float(
                    np.mean(
                        model_decode_itl
                    )
                    *
                    1000
                )

                if len(model_decode_itl)

                else np.nan
            ),

        "model_median_decode_itl_ms":
            (
                float(
                    np.median(
                        model_decode_itl
                    )
                    *
                    1000
                )

                if len(model_decode_itl)

                else np.nan
            ),

        "model_p95_decode_itl_ms":
            (
                float(
                    np.quantile(
                        model_decode_itl,
                        0.95,
                    )
                    *
                    1000
                )

                if len(model_decode_itl)

                else np.nan
            ),


        # ---------------------------
        # TOTAL
        # ---------------------------

        "overall_latency_seconds":
            overall_latency_seconds,


        "decode_tokens_per_second":
            (
                tokens_after_first
                /
                time_after_first

                if (
                    not np.isnan(
                        time_after_first
                    )
                    and
                    time_after_first > 0
                )

                else np.nan
            ),


        "end_to_end_tokens_per_second":
            (
                output_token_count
                /
                overall_latency_seconds

                if overall_latency_seconds > 0

                else np.nan
            ),
    }

In [ ]:
# CELL 8 — TorchAO APIs and model loaders
from torchao.quantization import quantize_,Int8WeightOnlyConfig,Int8DynamicActivationInt8WeightConfig

def load_w8():
    model=AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID,dtype=torch.bfloat16).to("cpu").eval()
    quantize_(model,Int8WeightOnlyConfig())
    return model

def load_a8w8():
    model=AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID,dtype=torch.bfloat16).to("cpu").eval()
    quantize_(model,Int8DynamicActivationInt8WeightConfig())
    return model

In [ ]:
# ============================================================
# CELL 9 — BENCHMARK ONE CANDIDATE
# ============================================================

def benchmark(name, loader):

    print("\n" + "=" * 90)
    print(name)
    print("=" * 90)

    gc.collect()

    process = psutil.Process(os.getpid())

    rss_before = (
        process.memory_info().rss
        /
        1024**2
    )


    # --------------------------------------------------------
    # LOAD + QUANTIZE
    # --------------------------------------------------------

    start = time.perf_counter()

    model = loader()

    load_seconds = (
        time.perf_counter()
        -
        start
    )


    gc.collect()


    rss_after = (
        process.memory_info().rss
        /
        1024**2
    )


    # --------------------------------------------------------
    # SIZE
    # --------------------------------------------------------

    footprint_mb = model_footprint_mb(
        model
    )

    tensor_size_mb = model_tensor_size_mb(
        model
    )


    print(
        "Load/quantize:",
        round(load_seconds, 3),
        "s"
    )

    print(
        "Model footprint:",
        round(footprint_mb, 2),
        "MB"
    )

    print(
        "Tensor storage estimate:",
        round(tensor_size_mb, 2),
        "MB"
    )

    print(
        "Process RSS increase:",
        round(
            rss_after - rss_before,
            2
        ),
        "MB"
    )


    # --------------------------------------------------------
    # WARMUP
    # --------------------------------------------------------

    print("\nWarmup...")

    _ = stream_generate(
        model,
        TEST_TEXTS["01_very_short"],
        max_new_tokens=64,
        show=False,
    )


    # --------------------------------------------------------
    # FIVE STREAMING TESTS
    # --------------------------------------------------------

    rows = []


    for test_name, text in TEST_TEXTS.items():

        print("\n" + "-" * 90)

        print(
            name,
            "|",
            test_name
        )

        print("-" * 90)

        print("\nSTREAM:\n")


        result = stream_generate(
            model,
            text,
            max_new_tokens=MAX_NEW_TOKENS,
            show=True,
        )


        print("\nMETRICS")

        print(
            "Input tokens:",
            result["input_tokens"]
        )

        print(
            "Output tokens:",
            result["output_tokens"]
        )

        print(
            "TTFT:",
            round(
                result["ttft_seconds"],
                4
            ),
            "s"
        )

        print(
            "Mean ITL:",
            round(
                result["mean_itl_ms"],
                2
            ),
            "ms"
        )

        print(
            "Median ITL:",
            round(
                result["median_itl_ms"],
                2
            ),
            "ms"
        )

        print(
            "P95 ITL:",
            round(
                result["p95_itl_ms"],
                2
            ),
            "ms"
        )

        print(
            "Overall latency:",
            round(
                result[
                    "overall_latency_seconds"
                ],
                4
            ),
            "s"
        )


        rows.append({

            "model":
                name,

            "test":
                test_name,

            "input_words":
                len(text.split()),

            "input_tokens":
                result["input_tokens"],

            "output_tokens":
                result["output_tokens"],


            # --------------------------------------------
            # SIZE
            # --------------------------------------------

            "model_footprint_mb":
                footprint_mb,

            "tensor_storage_mb":
                tensor_size_mb,

            "rss_load_delta_mb":
                rss_after - rss_before,


            # --------------------------------------------
            # PERFORMANCE
            # --------------------------------------------

            "load_seconds":
                load_seconds,

            "ttft_seconds":
                result["ttft_seconds"],

            "mean_itl_ms":
                result["mean_itl_ms"],

            "median_itl_ms":
                result["median_itl_ms"],

            "p95_itl_ms":
                result["p95_itl_ms"],

            "overall_latency_seconds":
                result[
                    "overall_latency_seconds"
                ],

            "decode_tokens_per_second":
                result[
                    "decode_tokens_per_second"
                ],

            "end_to_end_tokens_per_second":
                result[
                    "end_to_end_tokens_per_second"
                ],

            "output":
                result["output"],

            "itl_values_ms":
                result["itl_values_ms"],
        })


    dataframe = pd.DataFrame(
        rows
    )


    dataframe.to_json(
        REPORT_DIR
        /
        f"{name}_five_text_results.json",

        orient="records",

        indent=2,
    )


    del model

    gc.collect()


    return dataframe

In [ ]:
# ============================================================
# CELL 10 — RUN INT8 WEIGHT-ONLY
# ============================================================

W8 = benchmark(
    "TorchAO_INT8_WEIGHT_ONLY",
    load_w8,
)


display(
    W8.drop(
        columns=[
            "output",
            "itl_values_ms",
        ]
    )
)


TorchAO_INT8_WEIGHT_ONLY


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torchao/quantization/quant_api.py:1356: UserWarning: Config Deprecation: version 1 of Int8WeightOnlyConfig is deprecated and will no longer be supported in a future release, please use version 2, see https://github.com/pytorch/ao/issues/2752 for more details
  warnings.warn(


Load/quantize: 0.887 s
Model footprint: 146.73 MB
Tensor storage estimate: 146.73 MB
Process RSS increase: 113.05 MB

Warmup...

------------------------------------------------------------------------------------------
TorchAO_INT8_WEIGHT_ONLY | 01_very_short
------------------------------------------------------------------------------------------

STREAM:


-  Apex Systems reported quarterly revenue of $1.8 billion, up 9% year over year.
-  Operating profit increased 6% to $240 million, while management maintained its full-year guidance.

METRICS
Input tokens: 242
Output tokens: 40
TTFT: 1.3302 s
Mean ITL: 40.13 ms
Median ITL: 38.9 ms
P95 ITL: 48.17 ms
Overall latency: 2.8957 s

------------------------------------------------------------------------------------------
TorchAO_INT8_WEIGHT_ONLY | 02_short
------------------------------------------------------------------------------------------

STREAM:


-  Vertex Systems reported quarterly revenue of $3.8 billion, up 16% from the sa

,model,test,input_words,input_tokens,output_tokens,model_footprint_mb,tensor_storage_mb,rss_load_delta_mb,load_seconds,ttft_seconds,mean_itl_ms,median_itl_ms,p95_itl_ms,overall_latency_seconds,decode_tokens_per_second,end_to_end_tokens_per_second
0,TorchAO_INT8_WEIGHT_ONLY,01_very_short,26,242,40,146.729492,146.729492,113.046875,0.887397,1.330188,40.133918,38.899551,48.170400,2.895667,24.912506,13.813743
1,TorchAO_INT8_WEIGHT_ONLY,02_short,66,291,91,146.729492,146.729492,113.046875,0.887397,1.607737,49.632796,50.640101,58.338677,6.075129,20.145984,14.979106
2,TorchAO_INT8_WEIGHT_ONLY,03_medium,102,348,150,146.729492,146.729492,113.046875,0.887397,1.938722,42.242036,41.376085,48.493585,8.233401,23.670786,18.218474
3,TorchAO_INT8_WEIGHT_ONLY,04_long,150,402,139,146.729492,146.729492,113.046875,0.887397,2.705112,45.114752,42.092176,58.039291,8.931530,22.163628,15.562844
4,TorchAO_INT8_WEIGHT_ONLY,05_very_long,288,610,256,146.729492,146.729492,113.046875,0.887397,8.994637,65.251301,57.401191,161.197363,25.635528,15.323698,9.986141


In [ ]:
# CELL 11 — run dynamic A8W8
A8W8=benchmark("TorchAO_A8W8_DYNAMIC",load_a8w8)
display(A8W8.drop(columns=["output","itl_values_ms"]))


TorchAO_A8W8_DYNAMIC


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Load/quantize: 0.549 s
Model footprint: 146.73 MB
Tensor storage estimate: 146.73 MB
Process RSS increase: 131.63 MB

Warmup...

------------------------------------------------------------------------------------------
TorchAO_A8W8_DYNAMIC | 01_very_short
------------------------------------------------------------------------------------------

STREAM:


-  Apex Systems reported quarterly revenue of $1.8 billion, up 9% year over year.
-  Operating profit increased 6% to $240 million, while management maintained its full-year guidance.

METRICS
Input tokens: 242
Output tokens: 40
TTFT: 4.5906 s
Mean ITL: 71.56 ms
Median ITL: 70.57 ms
P95 ITL: 79.26 ms
Overall latency: 7.3818 s

------------------------------------------------------------------------------------------
TorchAO_A8W8_DYNAMIC | 02_short
------------------------------------------------------------------------------------------

STREAM:


-  Vertex Systems reported quarterly revenue of $3.8 billion, up 16% from the same peri

,model,test,input_words,input_tokens,output_tokens,model_footprint_mb,tensor_storage_mb,rss_load_delta_mb,load_seconds,ttft_seconds,mean_itl_ms,median_itl_ms,p95_itl_ms,overall_latency_seconds,decode_tokens_per_second,end_to_end_tokens_per_second
0,TorchAO_A8W8_DYNAMIC,01_very_short,26,242,40,146.729492,146.729492,131.628906,0.549132,4.590617,71.562931,70.570011,79.257638,7.381805,13.972545,5.418729
1,TorchAO_A8W8_DYNAMIC,02_short,66,291,91,146.729492,146.729492,131.628906,0.549132,6.923780,72.614523,71.257441,83.641357,13.459458,13.770568,6.761045
2,TorchAO_A8W8_DYNAMIC,03_medium,102,348,150,146.729492,146.729492,131.628906,0.549132,7.965698,101.254518,73.278143,333.579914,23.053186,9.875732,6.506693
3,TorchAO_A8W8_DYNAMIC,04_long,150,402,140,146.729492,146.729492,131.628906,0.549132,9.015194,79.784003,73.589994,117.475574,20.106053,12.532844,6.963077
4,TorchAO_A8W8_DYNAMIC,05_very_long,288,610,169,146.729492,146.729492,131.628906,0.549132,12.729958,101.220429,76.817390,206.590132,29.735648,9.879047,5.683414


In [ ]:
# CELL 12 — combined detailed results
RESULTS=pd.concat([W8,A8W8],ignore_index=True)
RESULTS.to_json(REPORT_DIR/"all_results.json",orient="records",indent=2)
display(RESULTS.drop(columns=["output","itl_values_ms"]))

,model,test,input_words,input_tokens,output_tokens,model_footprint_mb,tensor_storage_mb,rss_load_delta_mb,load_seconds,ttft_seconds,mean_itl_ms,median_itl_ms,p95_itl_ms,overall_latency_seconds,decode_tokens_per_second,end_to_end_tokens_per_second
0,TorchAO_INT8_WEIGHT_ONLY,01_very_short,26,242,40,146.729492,146.729492,113.046875,0.887397,1.330188,40.133918,38.899551,48.170400,2.895667,24.912506,13.813743
1,TorchAO_INT8_WEIGHT_ONLY,02_short,66,291,91,146.729492,146.729492,113.046875,0.887397,1.607737,49.632796,50.640101,58.338677,6.075129,20.145984,14.979106
2,TorchAO_INT8_WEIGHT_ONLY,03_medium,102,348,150,146.729492,146.729492,113.046875,0.887397,1.938722,42.242036,41.376085,48.493585,8.233401,23.670786,18.218474
3,TorchAO_INT8_WEIGHT_ONLY,04_long,150,402,139,146.729492,146.729492,113.046875,0.887397,2.705112,45.114752,42.092176,58.039291,8.931530,22.163628,15.562844
4,TorchAO_INT8_WEIGHT_ONLY,05_very_long,288,610,256,146.729492,146.729492,113.046875,0.887397,8.994637,65.251301,57.401191,161.197363,25.635528,15.323698,9.986141
5,TorchAO_A8W8_DYNAMIC,01_very_short,26,242,40,146.729492,146.729492,131.628906,0.549132,4.590617,71.562931,70.570011,79.257638,7.381805,13.972545,5.418729
6,TorchAO_A8W8_DYNAMIC,02_short,66,291,91,146.729492,146.729492,131.628906,0.549132,6.923780,72.614523,71.257441,83.641357,13.459458,13.770568,6.761045
7,TorchAO_A8W8_DYNAMIC,03_medium,102,348,150,146.729492,146.729492,131.628906,0.549132,7.965698,101.254518,73.278143,333.579914,23.053186,9.875732,6.506693
8,TorchAO_A8W8_DYNAMIC,04_long,150,402,140,146.729492,146.729492,131.628906,0.549132,9.015194,79.784003,73.589994,117.475574,20.106053,12.532844,6.963077
9,TorchAO_A8W8_DYNAMIC,05_very_long,288,610,169,146.729492,146.729492,131.628906,0.549132,12.729958,101.220429,76.817390,206.590132,29.735648,9.879047,5.683414


In [ ]:
# ============================================================
# CELL 13 — FINAL COMPARISON
# ============================================================

SUMMARY = (
    RESULTS
    .groupby(
        "model",
        as_index=False
    )
    .agg(

        model_footprint_mb=(
            "model_footprint_mb",
            "first"
        ),

        tensor_storage_mb=(
            "tensor_storage_mb",
            "first"
        ),

        rss_load_delta_mb=(
            "rss_load_delta_mb",
            "first"
        ),

        avg_ttft_seconds=(
            "ttft_seconds",
            "mean"
        ),

        median_ttft_seconds=(
            "ttft_seconds",
            "median"
        ),

        avg_mean_itl_ms=(
            "mean_itl_ms",
            "mean"
        ),

        avg_p95_itl_ms=(
            "p95_itl_ms",
            "mean"
        ),

        avg_overall_latency_seconds=(
            "overall_latency_seconds",
            "mean"
        ),

        avg_decode_tokens_per_second=(
            "decode_tokens_per_second",
            "mean"
        ),

        avg_end_to_end_tokens_per_second=(
            "end_to_end_tokens_per_second",
            "mean"
        ),
    )
)


display(
    SUMMARY
)

,model,model_footprint_mb,tensor_storage_mb,rss_load_delta_mb,avg_ttft_seconds,median_ttft_seconds,avg_mean_itl_ms,avg_p95_itl_ms,avg_overall_latency_seconds,avg_decode_tokens_per_second,avg_end_to_end_tokens_per_second
0,TorchAO_A8W8_DYNAMIC,146.729492,146.729492,131.628906,8.245049,7.965698,85.287281,164.108923,18.747230,12.006147,6.266592
1,TorchAO_INT8_WEIGHT_ONLY,146.729492,146.729492,113.046875,3.315279,1.938722,48.474961,74.847863,10.354251,21.243321,14.512061


In [ ]:
# CELL 14 — compare TTFT by text length
display(RESULTS.pivot(index="test",columns="model",values="ttft_seconds"))

model,TorchAO_A8W8_DYNAMIC,TorchAO_INT8_WEIGHT_ONLY
test,,
01_very_short,4.590617,1.330188
02_short,6.923780,1.607737
03_medium,7.965698,1.938722
04_long,9.015194,2.705112
05_very_long,12.729958,8.994637


In [ ]:
# CELL 15 — compare mean ITL by text length
display(RESULTS.pivot(index="test",columns="model",values="mean_itl_ms"))

model,TorchAO_A8W8_DYNAMIC,TorchAO_INT8_WEIGHT_ONLY
test,,
01_very_short,71.562931,40.133918
02_short,72.614523,49.632796
03_medium,101.254518,42.242036
04_long,79.784003,45.114752
05_very_long,101.220429,65.251301


In [ ]:
# CELL 16 — inspect every token gap for very-long input
for name in RESULTS["model"].unique():
    row=RESULTS[(RESULTS["model"]==name)&(RESULTS["test"]=="05_very_long")].iloc[0]
    gaps=np.array(row["itl_values_ms"])
    print("\n",name)
    print("gaps:",len(gaps),"median:",round(np.median(gaps),2),
          "p95:",round(np.quantile(gaps,.95),2),"max:",round(np.max(gaps),2),"ms")


 TorchAO_INT8_WEIGHT_ONLY
gaps: 255 median: 57.4 p95: 161.2 max: 244.25 ms

 TorchAO_A8W8_DYNAMIC
gaps: 168 median: 76.82 p95: 206.59 max: 442.44 ms


In [ ]:
# CELL 17 — files
print("ARTIFACTS")
for p in ARTIFACT_DIR.iterdir():
    if p.is_dir():print(p.name,round(dir_mb(p),2),"MB")
print("\nREPORTS")
for p in REPORT_DIR.glob("*"):print(p.name)

ARTIFACTS
TorchAO_INT8_WEIGHT_ONLY 0.0 MB

REPORTS
TorchAO_INT8_WEIGHT_ONLY_five_text_results.json
TorchAO_A8W8_DYNAMIC_five_text_results.json
all_results.json


## Interpretation

- **TTFT** measures how long the user waits before streaming starts. For T5 it includes the full encoder pass plus the first decoder step.
- **ITL** measures the gaps between later generated tokens. Those decoder steps reuse `past_key_values`.
- **Overall latency** measures completion of the entire summary.
- **Actual artifact size** is measured from files produced by `save_pretrained()`, not from a theoretical bits-per-parameter calculation.

For interactive serving, TTFT and p95 ITL usually matter more to perceived responsiveness than total latency alone.